In [ ]:
!pip install kagglehub[pandas-datasets]

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
amananandrai_clickbait_dataset_path = kagglehub.dataset_download('amananandrai/clickbait-dataset')

print('Data source import complete.')


100%|██████████| 743k/743k [00:00<00:00, 862kB/s]

Extracting files...
Data source import complete.


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
! ls /kaggle/

input


In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amananandrai/clickbait-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'clickbait-dataset' dataset.
Path to dataset files: /kaggle/input/clickbait-dataset


In [5]:
# load the data in the cwd
! ls {path}
! cp {path}/clickbait_data.csv .

clickbait_data.csv


In [6]:
df = pd.read_csv("clickbait_data.csv")
df.head()

,headline,clickbait
0,Should I Get Bings,1
1,Which TV Female Friend Group Do You Belong In,1
2,"The New ""Star Wars: The Force Awakens"" Trailer...",1
3,"This Vine Of New York On ""Celebrity Big Brothe...",1
4,A Couple Did A Stunning Photo Shoot With Their...,1


## Data Preprocessing

In [7]:
# shuffle the df
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
df_shuffled.head()

,headline,clickbait
0,Filipino activist arrested for disrupting Mani...,0
1,"International Board fixes soccer field size, h...",0
2,24 Rules For Women On A First Date With A Man,1
3,Political fallout from the sacking of Professo...,0
4,"Which ""Clueless"" Character Are You Based On Yo...",1


In [8]:
df_shuffled.describe(include='object')

,headline
count,32000
unique,32000
top,"With Late Push, Johnson Edges Hamlin at Martin..."
freq,1


In [9]:
df_shuffled.isnull().sum()

,0
headline,0
clickbait,0


In [10]:
df_shuffled.rename(columns={'headline': 'input'}, inplace=True)
df_shuffled.head()

,input,clickbait
0,Filipino activist arrested for disrupting Mani...,0
1,"International Board fixes soccer field size, h...",0
2,24 Rules For Women On A First Date With A Man,1
3,Political fallout from the sacking of Professo...,0
4,"Which ""Clueless"" Character Are You Based On Yo...",1


## Tokenization

In [11]:
from datasets import Dataset, DatasetDict

ds = Dataset.from_pandas(df_shuffled)

In [12]:
ds

Dataset({
    features: ['input', 'clickbait'],
    num_rows: 32000
})

In [13]:
# lets choose the model
model_nm = "microsoft/deberta-v3-small"

In [14]:
# get the autotokenizer for our model
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tz = AutoTokenizer.from_pretrained(model_nm)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [15]:
# simple function which tokenizes our inputs
def tok_fun(x):
    return tz(x["input"])

In [16]:
# use map for parallel processing

tok_ds = ds.map(tok_fun, batched=True)

Map:   0%|          | 0/32000 [00:00<?, ? examples/s]

In [17]:
# the tokenization adds a new col input_ids
row = tok_ds[0]
row["input"], row["input_ids"]

('Filipino activist arrested for disrupting Manila Cathedral mass in Reproductive Health Bill protest',
 [1,
  18797,
  9664,
  3740,
  270,
  28964,
  16385,
  11895,
  2675,
  267,
  43490,
  1516,
  2492,
  5933,
  2])

In [18]:
# transformers always use labels as the predicting value, hence rename
tok_ds = tok_ds.rename_columns({'clickbait': 'labels'})

In [19]:
tok_ds

Dataset({
    features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 32000
})

In [20]:
# split into train and val
dds = tok_ds.train_test_split(0.25, seed=42)
dds

DatasetDict({
    train: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 24000
    })
    test: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8000
    })
})

## Training the model

In [21]:
from transformers import TrainingArguments, Trainer

In [22]:
# fix the main params
from sklearn.metrics import f1_score

bs = 128
lr = 1e-5
epochs = 5

def compute_metrics(eval_pred):
    p, y = eval_pred
    return {'f1_score': f1_score(y, p.argmax(-1))}


In [23]:
# define the model arguments

args = TrainingArguments('outputs', learning_rate=lr, warmup_ratio=0.1, lr_scheduler_type='cosine', fp16=True,
    eval_strategy="epoch", per_device_train_batch_size=bs, per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs, weight_decay=0.01, report_to='none')

In [24]:
# define the model

model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels=2)
trainer = Trainer(model, args, train_dataset=dds['train'], eval_dataset=dds['test'],
                  tokenizer=tz, compute_metrics=compute_metrics)

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-4221974911.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model, args, train_dataset=dds['train'], eval_dataset=dds['test'],


In [25]:
trainer.train();

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,F1 Score
1,No log,0.022544,0.992991
2,No log,0.018347,0.995011
3,0.124800,0.017183,0.996002
4,0.124800,0.017150,0.996376
5,0.124800,0.017368,0.996248


We got around 99 percent F1 Score which is a combination of precision and recall!

## Try out on test set

In [27]:
test_csv = """
headline,clickbait
"10 Ways To Instantly Improve Your Mood",1
"Government Releases New Economic Policy Report",0
"Can You Guess Which Celebrity Said This?",1
"How To Bake The Perfect Chocolate Cake",1
"Scientists Discover New Method To Purify Water",0
"""

with open('test.csv', 'w') as f:
    f.write(test_csv)

In [28]:
df_test = pd.read_csv("test.csv")
df_test.head()

,headline,clickbait
0,10 Ways To Instantly Improve Your Mood,1
1,Government Releases New Economic Policy Report,0
2,Can You Guess Which Celebrity Said This?,1
3,How To Bake The Perfect Chocolate Cake,1
4,Scientists Discover New Method To Purify Water,0


In [32]:
def preprocess(df):
    ds = Dataset.from_pandas(df)
    ds = ds.rename_columns({'headline': 'input'})
    tok_ds = ds.map(tok_fun, batched=True)
    tok_ds = tok_ds.rename_columns({'clickbait': 'labels'})
    return tok_ds

In [33]:
eval_ds = preprocess(df_test)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [41]:
preds = trainer.predict(eval_ds)
df_test["Predicted"] = preds[1]
df_test.head()

,headline,clickbait,Predicted
0,10 Ways To Instantly Improve Your Mood,1,1
1,Government Releases New Economic Policy Report,0,0
2,Can You Guess Which Celebrity Said This?,1,1
3,How To Bake The Perfect Chocolate Cake,1,1
4,Scientists Discover New Method To Purify Water,0,0
